In [2]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")
            
            
# 데이트타임형으로 변환 및 기간 전처리
def set_datetime(df, column):
    df[column] =  pd.to_datetime(df[column])
    print(f'✅ {column}데이트 타입 형변환 및 기간 전처리 완료')
    
    

            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


### events: 유저들이 포인트를 얻을수 있었던 이벤트종류

In [47]:
events = get_df('votes', 'events')
set_datetime(events, 'created_at')
events.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,title,plus_point,event_type,is_expired,created_at
0,1,코드잇 은행 가입 이벤트,500,FCFS,1,2023-06-20 11:56:38
1,2,코드잇 멤버십 가입 이벤트,1000,FCFS,1,2023-08-08 07:43:45
2,3,예고 영상 기대평 이벤트,500,FCFS,1,2023-09-24 17:05:59


### 15. event_receipts: 유저들이 포인트 이벤트에 참여했던 기록

In [46]:
event_receipts = get_df('votes', 'event_receipts')
event_receipts = event_receipts[['id' ,'user_id' ,'event_id' ,'plus_point' ,'created_at']]
set_datetime(event_receipts, 'created_at')
event_receipts = event_receipts.drop(81)
event_receipts.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,user_id,event_id,plus_point,created_at
0,2,1193618,1,500,2023-06-22 09:25:16
1,3,928351,1,500,2023-06-22 09:38:53
2,4,904872,1,500,2023-06-22 10:32:15
3,5,974697,1,500,2023-06-22 13:03:06
4,6,1168260,1,500,2023-06-22 13:40:38


In [ ]:
# # 날자 파생 컬럼 추가
# event_receipts['C_y_m_d'] = event_receipts['created_at'].dt.to_period('D').astype('str')
# event_receipts['C_y_m'] = event_receipts['created_at'].dt.to_period('M').astype('str')
# event_receipts['C_hour'] = event_receipts['created_at'].dt.hour
# event_receipts['C_weekday'] = event_receipts['created_at'].dt.weekday
# event_receipts['C_is_weekend'] = event_receipts['created_at'].dt.weekday >= 5
# event_receipts['C_time_of_day'] = pd.cut(event_receipts['created_at'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

In [41]:
# event_id_C_y_m_d = event_receipts.groupby(['event_id'])['C_y_m_d'].value_counts().reset_index().sort_values(by=['event_id', 'C_y_m_d'])

# # 별로 나누기
# event1 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 1]
# event2 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 2]
# event3 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 3]

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=event1['C_y_m_d'],
#     y=event1['count'],
#     mode='lines+markers',
#     name='Weekday',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=event2['C_y_m_d'],
#     y=event2['count'],
#     mode='lines+markers',
#     name='Weekend',
#     line=dict(color='orange')
# ))

# # event3 라인
# fig.add_trace(go.Scatter(
#     x=event3['C_y_m_d'],
#     y=event3['count'],
#     mode='lines+markers',
#     name='Weekend',
#     line=dict(color='purple')
# ))

# fig.show()

In [ ]:
# event_id_C_hour = event_receipts.groupby(['event_id'])['C_hour'].value_counts().reset_index().sort_values(by=['event_id', 'C_hour'])
# event1 = event_id_C_hour.query('event_id == 1')
# event2 = event_id_C_hour.query('event_id == 2')
# event3 = event_id_C_hour.query('event_id == 3')

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=event1['C_hour'],
#     y=event1['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=event2['C_hour'],
#     y=event2['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='orange')
# ))

# # event3 라인ㅁ
# fig.add_trace(go.Scatter(
#     x=event3['C_hour'],
#     y=event3['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='purple')
# ))
# fig.update_layout(
#     title='이벤트별 참가시간대',
#     xaxis_title='Hour of Day',
#     yaxis_title='Count',
#     xaxis=dict(tickmode='linear', dtick=1),
#     template='plotly_white',
#     width=1000,
#     height=500
# )

# fig.show()

In [ ]:
# from datetime import timedelta
# kst = event_receipts[['event_id', 'created_at']]
# kst['kst_created_at'] = pd.to_datetime(kst['created_at'], utc=True) + timedelta(hours=9)

# kst['C_y_m_d'] = kst['kst_created_at'].dt.to_period('D').astype('str')
# kst['C_y_m'] = kst['kst_created_at'].dt.to_period('M').astype('str')
# kst['C_hour'] = kst['kst_created_at'].dt.hour
# kst['C_weekday'] = kst['kst_created_at'].dt.weekday
# kst['C_is_weekend'] = kst['kst_created_at'].dt.weekday >= 5
# kst['C_time_of_day'] = pd.cut(kst['kst_created_at'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

In [ ]:
# kst_event_id_C_hour = kst.groupby(['event_id'])['C_hour'].value_counts().reset_index().sort_values(by=['event_id', 'C_hour'])
# kst_event1 = kst_event_id_C_hour.query('event_id == 1')
# kst_event2 = kst_event_id_C_hour.query('event_id == 2')
# kst_event3 = kst_event_id_C_hour.query('event_id == 3')

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=kst_event1['C_hour'],
#     y=kst_event1['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=kst_event2['C_hour'],
#     y=kst_event2['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='orange')
# ))

# # event3 라인ㅁ
# fig.add_trace(go.Scatter(
#     x=kst_event3['C_hour'],
#     y=kst_event3['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='purple')
# ))
# fig.update_layout(
#     title='이벤트별 참가시간대',
#     xaxis_title='Hour of Day',
#     yaxis_title='Count',
#     xaxis=dict(tickmode='linear', dtick=1),
#     template='plotly_white',
#     width=1000,
#     height=500
# )

# fig.show()

### accounts_paymenthistory: 포인트 구매기록

In [45]:
accounts_paymenthistory = get_df('votes', 'accounts_paymenthistory')
accounts_paymenthistory.head()

,id,productId,phone_type,created_at,user_id
0,6,heart.777,A,2023-05-13 21:28:34,1211127
1,7,heart.777,A,2023-05-13 21:29:39,1151343
2,8,heart.777,A,2023-05-13 21:31:33,1002147
3,9,heart.777,A,2023-05-13 21:31:39,1095040
4,11,heart.777,A,2023-05-13 21:34:32,1164081


### accounts_failpaymenthistory: 포인트 결제 실패 기록

In [44]:
accounts_failpaymenthistory = get_df('votes', 'accounts_failpaymenthistory')
accounts_failpaymenthistory.head()

,id,productId,phone_type,created_at,user_id
0,6,heart.200,A,2023-05-14 05:49:22,1055891
1,7,heart.777,A,2023-05-14 08:17:21,1152151
2,8,heart.777,A,2023-05-14 10:11:46,986200
3,9,heart.1000,A,2023-05-14 11:53:09,1028261
4,10,heart.777,A,2023-05-14 12:30:47,1235730


### accounts_pointhistory: 포인트 충전 및 사용에 대한 기록

In [133]:
accounts_pointhistory = get_df('votes', 'accounts_pointhistory')
accounts_pointhistory = accounts_pointhistory[['id', 'user_id', 'user_question_record_id', 'delta_point', 'created_at']]
accounts_pointhistory.head()

,id,user_id,user_question_record_id,delta_point,created_at
0,790629,849436,771777.0,9,2023-04-28 12:27:49
1,790652,849436,771800.0,9,2023-04-28 12:28:02
2,790664,849436,771812.0,5,2023-04-28 12:28:09
3,790680,849436,771828.0,13,2023-04-28 12:28:16
4,790703,849436,771851.0,5,2023-04-28 12:28:26


In [109]:
accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated().sum()
accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated(keep=False).sum()

0

In [108]:
accounts_pointhistory = accounts_pointhistory[~accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated()]

In [115]:
accounts_pointhistory[accounts_pointhistory['delta_point'] > 0]['delta_point'].unique()

array([   9,    5,   13,   10,   12,   15,   14,    8,    6,   11,    7,
        200,  500, 1000,  300,  210,  240,  220,  230,  250,  777,  260,
        270,  280,  100,   50,   60,   70,   80,   90,  110,  120,  130,
        140,  150,  160,  170,  180])

In [132]:
mask = pd.to_numeric(accounts_pointhistory['user_question_record_id'], errors='coerce').isnull()
accounts_pointhistory[mask]['delta_point'].value_counts()


delta_point
 200     1368
 500      429
 50       322
 1000     233
 300      180
 210      127
 220       64
 60        47
 230       36
 240       26
 250       20
 777       19
 70        16
 260       14
 270       12
 80        10
 100       10
 280       10
 90         7
 110        6
 120        6
 130        5
 140        4
 150        4
 160        4
 170        3
 180        3
-30         1
Name: count, dtype: int64

In [ ]:
accounts_pointhistory[accounts_pointhistory['delta_point'] > 0]['user_question_record_id'].unique().tolist()

[771777.0,
 771800.0,
 771812.0,
 771828.0,
 771851.0,
 771864.0,
 771894.0,
 771908.0,
 771912.0,
 771927.0,
 771940.0,
 771952.0,
 771966.0,
 771969.0,
 771974.0,
 771981.0,
 771995.0,
 772001.0,
 772002.0,
 772023.0,
 772025.0,
 772027.0,
 772030.0,
 772040.0,
 772044.0,
 772054.0,
 772056.0,
 772057.0,
 772080.0,
 772088.0,
 772098.0,
 772101.0,
 772106.0,
 772122.0,
 772123.0,
 772126.0,
 772128.0,
 772133.0,
 772140.0,
 772147.0,
 772150.0,
 772151.0,
 772157.0,
 772167.0,
 772168.0,
 772179.0,
 772184.0,
 772190.0,
 772195.0,
 772203.0,
 772214.0,
 772223.0,
 772225.0,
 772226.0,
 772233.0,
 772241.0,
 772246.0,
 772258.0,
 772268.0,
 772277.0,
 772278.0,
 772279.0,
 772289.0,
 772293.0,
 772297.0,
 772299.0,
 772300.0,
 772301.0,
 772309.0,
 772314.0,
 772315.0,
 772320.0,
 772332.0,
 772334.0,
 772343.0,
 772346.0,
 772359.0,
 772360.0,
 772364.0,
 772370.0,
 772386.0,
 772389.0,
 772395.0,
 772398.0,
 772414.0,
 772416.0,
 772426.0,
 772434.0,
 772439.0,
 772453.0,
 772457.0,

### accounts_userquestionrecord: 투표기록 테이블

In [ ]:
accounts_userquestionrecord = get_df('votes', 'accounts_userquestionrecord')
accounts_userquestionrecord = accounts_userquestionrecord[['id', 'user_id', 'chosen_user_id', 'question_id', 'question_piece_id', \
    'status' ,'answer_status', 'answer_updated_at', 'has_read', 'opened_times', 'report_count', 'created_at']]

,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at
0,771777,849436,849469,252,998458,닫힘,N,2023-04-28 12:27:49,0,0,0,2023-04-28 12:27:49
1,771800,849436,849446,244,998459,닫힘,N,2023-04-28 12:28:02,0,0,0,2023-04-28 12:28:02
2,771812,849436,849454,183,998460,닫힘,N,2023-04-28 12:28:09,1,0,0,2023-04-28 12:28:09
3,771828,849436,847375,101,998461,닫힘,N,2023-04-28 12:28:16,0,0,0,2023-04-28 12:28:16
4,771851,849436,849477,209,998462,닫힘,N,2023-04-28 12:28:26,1,0,0,2023-04-28 12:28:26


In [71]:
accounts_userquestionrecord['status'] = accounts_userquestionrecord['status'].replace({'C':'닫힘'}).replace({'I':'초성열림'}).replace({'B':'차단'})
accounts_userquestionrecord['answer_status'] = accounts_userquestionrecord['answer_status'].replace({'N':'미답변'}).replace({'P':'비공개'}).replace({'A':'공개'})
set_datetime(accounts_userquestionrecord, 'answer_updated_at')
set_datetime(accounts_userquestionrecord, 'created_at')
accounts_userquestionrecord['diff_time'] = accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']
accounts_userquestionrecord['diff_sec_time'] = (accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']).dt.total_seconds()

✅ answer_updated_at데이트 타입 형변환 및 기간 전처리 완료
✅ created_at데이트 타입 형변환 및 기간 전처리 완료


In [72]:
accounts_userquestionrecord[accounts_userquestionrecord.loc[:, ['user_id', 'chosen_user_id' ,'created_at']].duplicated(keep=False)].sort_values(
    by=['user_id', 'chosen_user_id' ,'created_at']
).head()

,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at,diff_time,diff_sec_time
497599,41415258,850096,865629,269,15408091,닫힘,미답변,2023-05-14 04:33:44,1,0,0,2023-05-14 04:33:44,0 days,0.0
497602,41415309,850096,865629,552,15408093,닫힘,미답변,2023-05-14 04:33:44,1,0,0,2023-05-14 04:33:44,0 days,0.0
497557,41411486,855914,867773,390,28683031,닫힘,미답변,2023-05-14 04:33:15,0,0,0,2023-05-14 04:33:15,0 days,0.0
497558,41411496,855914,867773,222,28683036,닫힘,미답변,2023-05-14 04:33:15,0,0,0,2023-05-14 04:33:15,0 days,0.0
470004,38408532,880602,876643,500,49921251,닫힘,미답변,2023-05-13 15:10:36,1,0,0,2023-05-13 15:10:36,0 days,0.0


In [65]:
accounts_userquestionrecord['status'].unique()

array(['닫힘', '초성열림', '차단'], dtype=object)

In [73]:
accounts_userquestionrecord.groupby('answer_status')['diff_sec_time'].mean().reset_index()

,answer_status,diff_sec_time
0,공개,32711.906407
1,미답변,-0.001299
2,비공개,29461.689383


In [76]:
accounts_userquestionrecord.groupby('status')['diff_sec_time'].mean().reset_index()

,status,diff_sec_time
0,닫힘,3077.741957
1,차단,994.922492
2,초성열림,5416.400987


In [80]:
accounts_userquestionrecord.groupby(['has_read', 'answer_status'])['status'].value_counts().reset_index()

,has_read,answer_status,status,count
0,0,미답변,닫힘,537672
1,0,미답변,초성열림,3629
2,0,미답변,차단,326
3,1,공개,닫힘,101080
4,1,공개,초성열림,10634
5,1,공개,차단,47
6,1,미답변,닫힘,511087
7,1,미답변,초성열림,44937
8,1,미답변,차단,281
9,1,비공개,닫힘,6483


In [75]:
accounts_userquestionrecord.groupby('answer_status')['status'].value_counts().reset_index()

,answer_status,status,count
0,공개,닫힘,101080
1,공개,초성열림,10634
2,공개,차단,47
3,미답변,닫힘,1048759
4,미답변,초성열림,48566
5,미답변,차단,607
6,비공개,닫힘,6483
7,비공개,초성열림,1378
8,비공개,차단,4


In [ ]:
보낸 유저이름 정확히 보여